# NB-07_salida_cg_imputa

Process Flow SAS: **Salida CG Imputa** — `ProcessFlowContainer-diHU2ZUCEv1lqT51`

In [ ]:
# ========= Parámetros =========
# Variables macro del SAS original. El .egp NO las define (venían del
# entorno SAS): su valor sale de la entrevista B4 o de
# project_config.yaml → run.macro_params, o se inyecta acá
# (celda 'parameters' de papermill).

Year = None  # &Year — nadie declaró su valor

faltantes = [n for n, v in {"Year": Year}.items() if v is None]
if faltantes:
    raise ValueError(f"Parámetros sin valor: {faltantes}")

In [ ]:
# ========= Celda 1: Configuración =========
import pandas as pd
import numpy as np
import os
import sqlalchemy
import datetime
from pathlib import Path
from sqlalchemy import text

# Conexión a BD — la define el orquestador al ejecutar
engine = sqlalchemy.create_engine(os.environ["SASMIG_DB_URL"], pool_pre_ping=True)
# Sesión de BD del notebook — espejo de la sesión WORK de SAS: las
# tablas temporales #tmp viven en ESTA conexión y mueren al cerrar el
# kernel. AUTOCOMMIT: cada statement commitea, como los pasos de SAS.
work_conn = engine.connect().execution_options(isolation_level="AUTOCOMMIT")

# Logging liviano de resultados — aprobado en la entrevista (Fase 4)
_LOG_PATH = Path("log") / "NB-07_salida_cg_imputa.log"
_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
def _log(label, value=None):
    """Una línea por celda: imprime y persiste. Jamás rompe la corrida."""
    try:
        if hasattr(value, "shape"):
            detail = f"{value.shape[0]} filas x {value.shape[1]} cols"
        elif isinstance(value, int):
            detail = f"{value} filas"
        elif value is None:
            detail = "ok"
        else:
            detail = str(value)
        line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {label}: {detail}"
        print(line)
        with open(_LOG_PATH, "a", encoding="utf-8") as fh:
            fh.write(line + "\n")
    except Exception:
        pass  # el log nunca puede tumbar el notebook
with open(_LOG_PATH, "a", encoding="utf-8") as _fh:
    _fh.write(f"\n=== corrida {datetime.datetime.now():%Y-%m-%d %H:%M:%S} ===\n")


## 1_Cobertura

Identifica entidades y monedas cuya ejecuci\u00f3n informada no alcanza el \u00faltimo mes disponible del a\u00f1o, con el detalle de partida, cap\u00edtulo y programa a revisar

*confianza: medium · verificador: approve · SAS: PROC SQL (CREATE TABLE / UPDATE / DROP) + PROC TRANSPOSE + PROC SORT NODUPKEY*

In [ ]:
# ========= 1_Cobertura =========
# /*i. DETERMINA FALTANTES EN INFO CONTRALORIA*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cgr"))
# Year interpolado como entero: un bind pasa por sp_prepexec y la #tmp no sobrevive
sql_ejec_cgr = f"""
SELECT CONCAT_WS('_', A\u00d1O, MES) AS PERIODO,
       MONEDA, A\u00d1O, MES, C_ENTIDAD,
       LTRIM(CAST(C_ENTIDAD AS varchar(10))) AS ENTIDAD,
       C_PARTIDA, C_CAPITULO, C_PROGRAMA,
       CONCAT_WS('_', C_PARTIDA, C_CAPITULO, C_PROGRAMA) AS PCP,
       CONCAT_WS('_', N_PARTIDA, N_CAPITULO, N_PROGRAMA) AS N_PCP,
       SUM(DEVENG) / 1000.0 AS DATO,
       CASE WHEN A\u00d1O = {int(Year)} THEN 'Pt' ELSE 'Pt_1' END AS TIEMPO
INTO #ejec_cgr
FROM GOBGENER.dbo.EJECUCION
WHERE A\u00d1O = {int(Year)} OR (A\u00d1O = {int(Year) - 1} AND MES = 12)
GROUP BY A\u00d1O, MES, MONEDA, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA,
         N_PARTIDA, N_CAPITULO, N_PROGRAMA
"""
work_conn.execute(text(sql_ejec_cgr))


In [ ]:
# UPDATE EJEC_CGR SET DATO=. WHERE DATO IN (0,.,) -> el faltante de SAS es NULL en SQL Server
res = work_conn.execute(text("UPDATE #ejec_cgr SET DATO = NULL WHERE DATO = 0 OR DATO IS NULL"))
_log("UPDATE #ejec_cgr DATO->NULL", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #cuenta_meses"))
# COUNT(DATO) cuenta solo los meses con dato (no NULL), igual que COUNT(t1.dato) en SAS
sql_cuenta_meses = """
SELECT t1.TIEMPO, t1.C_ENTIDAD, t1.MONEDA, COUNT(t1.DATO) AS N
INTO #cuenta_meses
FROM #ejec_cgr t1
GROUP BY t1.TIEMPO, t1.C_ENTIDAD, t1.MONEDA
"""
work_conn.execute(text(sql_cuenta_meses))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #max_mes"))
sql_max_mes = """
SELECT TIEMPO, MAX(MES) AS MAX_MES
INTO #max_mes
FROM #ejec_cgr
WHERE TIEMPO = 'Pt'
GROUP BY TIEMPO
"""
work_conn.execute(text(sql_max_mes))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #max_mes_ent"))
# dato<>. en SAS = tiene dato: en SQL Server es DATO IS NOT NULL
sql_max_mes_ent = """
SELECT TIEMPO, C_ENTIDAD, MONEDA, MAX(MES) AS MAX_MES
INTO #max_mes_ent
FROM #ejec_cgr
WHERE TIEMPO = 'Pt' AND DATO IS NOT NULL
GROUP BY TIEMPO, C_ENTIDAD, MONEDA
"""
work_conn.execute(text(sql_max_mes_ent))


In [ ]:
# PROC TRANSPOSE DATA=Cuenta_Meses OUT=TRANS_MES (DROP=_NAME_); BY C_ENTIDAD moneda; ID TIEMPO;
cuenta_meses = pd.read_sql(
    text("SELECT C_ENTIDAD, MONEDA, TIEMPO, N FROM #cuenta_meses ORDER BY C_ENTIDAD, MONEDA, TIEMPO"),
    work_conn,
)
trans_mes = (
    cuenta_meses.pivot_table(
        index=["C_ENTIDAD", "MONEDA"], columns="TIEMPO", values="N", aggfunc="first"
    )
    .reset_index()
    .rename_axis(columns=None)
)
for _c in ["Pt", "Pt_1"]:
    if _c not in trans_mes.columns:
        trans_mes[_c] = np.nan
trans_mes = trans_mes[["C_ENTIDAD", "MONEDA", "Pt", "Pt_1"]]
trans_mes.to_sql("#trans_mes", work_conn, if_exists="replace", index=False)
_log("trans_mes", trans_mes)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #cobertura_mes"))
# /*Detecta cobertura faltante*/
# En SAS el faltante es menor que cualquier numero: Pt sin meses informados => "Revisar"
sql_cobertura_mes = """
SELECT T1.C_ENTIDAD, T1.MONEDA, T1.Pt, T1.Pt_1, T2.MAX_MES,
       CASE WHEN T1.Pt IS NULL OR T1.Pt < T2.MAX_MES THEN 'Revisar' ELSE 'OK' END AS ID
INTO #cobertura_mes
FROM #trans_mes T1
CROSS JOIN #max_mes T2
"""
work_conn.execute(text(sql_cobertura_mes))


In [ ]:
# /*Define nombre de entidades y c\u00f3digos claves*/
work_conn.execute(text("DROP TABLE IF EXISTS #detalle_entidad"))
sql_detalle_entidad = """
SELECT DISTINCT T1.MONEDA, T1.C_ENTIDAD, T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA, T1.N_PCP
INTO #detalle_entidad
FROM #ejec_cgr T1
"""
work_conn.execute(text(sql_detalle_entidad))


In [ ]:
# PROC SORT ... OUT=DETALLE_ENTIDAD_UN nodupkey; by MONEDA C_ENTIDAD;
# una tabla SQL no tiene orden: se ordena explicito para que el registro conservado sea determinista
detalle_entidad = pd.read_sql(
    text(
        "SELECT MONEDA, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, N_PCP FROM #detalle_entidad "
        "ORDER BY MONEDA, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA"
    ),
    work_conn,
)
detalle_entidad_un = detalle_entidad.drop_duplicates(
    subset=["MONEDA", "C_ENTIDAD"], keep="first"
).reset_index(drop=True)
detalle_entidad_un.to_sql("#detalle_entidad_un", work_conn, if_exists="replace", index=False)
_log("detalle_entidad_un", detalle_entidad_un)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #cobertura_detalle_base"))
# /*FALTANTES CON DETALLE*/
# max_mes>T3.MAX_MES: en SAS un MAX_MES_ENT faltante (entidad sin ningun mes con dato)
# es menor que cualquier mes, asi que la fila entra; en SQL Server hay que decirlo
sql_cobertura_detalle = f"""
SELECT DISTINCT T1.MONEDA, {int(Year)} AS A\u00d1O, T1.C_ENTIDAD, T1.C_PARTIDA, T1.C_CAPITULO,
       T1.C_PROGRAMA, T1.N_PCP, T2.ID, T2.MAX_MES, T3.MAX_MES AS MAX_MES_ENT
INTO #cobertura_detalle_base
FROM #detalle_entidad_un T1
LEFT JOIN #cobertura_mes T2 ON (T1.C_ENTIDAD = T2.C_ENTIDAD AND T1.MONEDA = T2.MONEDA)
LEFT JOIN #max_mes_ent T3 ON (T1.C_ENTIDAD = T3.C_ENTIDAD AND T1.MONEDA = T3.MONEDA)
WHERE (T2.ID = 'Revisar' AND (T3.MAX_MES IS NULL OR T2.MAX_MES > T3.MAX_MES))
   OR (T2.ID = 'Revisar' AND T2.Pt IS NULL)
"""
work_conn.execute(text(sql_cobertura_detalle))


In [ ]:
# DATA COBERTURA_DETALLE; SET COBERTURA_DETALLE; C_PARTIDA_2=C_PARTIDA;
# T-SQL no reescribe una tabla desde si misma: la version previa queda en #cobertura_detalle_base
work_conn.execute(text("DROP TABLE IF EXISTS #cobertura_detalle"))
work_conn.execute(
    text("SELECT *, C_PARTIDA AS C_PARTIDA_2 INTO #cobertura_detalle FROM #cobertura_detalle_base")
)
work_conn.execute(text("DROP TABLE IF EXISTS #cobertura_detalle_base"))


In [ ]:
# UPDATE COBERTURA_DETALLE SET C_PARTIDA_2='32' WHERE C_PARTIDA='62' AND A\u00d1O>=2025
res = work_conn.execute(
    text("UPDATE #cobertura_detalle SET C_PARTIDA_2 = '32' WHERE C_PARTIDA = '62' AND A\u00d1O >= 2025")
)
_log("UPDATE #cobertura_detalle C_PARTIDA_2", res.rowcount)


In [ ]:
cobertura_detalle = pd.read_sql(text("SELECT * FROM #cobertura_detalle"), work_conn)
_log("cobertura_detalle", cobertura_detalle)


In [ ]:
# PROC SQL; DROP TABLE EJEC_CGR,CUENTA_MESES,MAX_MES,MAX_MES_ENT,TRANS_MES,DETALLE_ENTIDAD;
for _t in ["#ejec_cgr", "#cuenta_meses", "#max_mes", "#max_mes_ent", "#trans_mes", "#detalle_entidad"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## 2a_TCons Mixtas_M_CGR_Faltante

Completa la ejecución presupuestaria faltante con información de Contraloría (incluida la imputación de gobiernos regionales al mes DIPRES) y redistribuye las transacciones mixtas —muestra y expansión— por clasificador de cuentas nacionales para el año en proceso

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE encadenados (joins + agregación) y DATA step SET/DROP*

In [ ]:
# ========= 2a_TCons Mixtas_M_CGR_Faltante =========
# /*Ejecucion cgr del periodo, para faltantes*/
# COBERTURA_DETALLE lo deja el nodo 1_Cobertura en memoria: se materializa en la sesion
cobertura_detalle.to_sql("#cobertura_detalle", work_conn, if_exists="replace", index=False)
_log("#cobertura_detalle", cobertura_detalle)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cgr_i"))
sql_ejec_cgr_i = """
SELECT T1.*
INTO #ejec_cgr_i
FROM GOBGENER.dbo.EJECUCION T1
LEFT JOIN #cobertura_detalle T3
       ON (T1.AÑO = T3.AÑO - 1) AND (T1.C_ENTIDAD = T3.C_ENTIDAD)
      AND (T1.C_PARTIDA = T3.C_PARTIDA_2) AND (T1.C_CAPITULO = T3.C_CAPITULO)
      AND (T1.C_PROGRAMA = T3.C_PROGRAMA) AND (T1.MONEDA = T3.MONEDA)
WHERE T3.ID = 'Revisar' AND T1.MES <= T3.MAX_MES AND T1.MES > T3.MAX_MES_ENT
  AND T3.C_PARTIDA IN ('88','62','31') AND T3.C_PARTIDA_2 IN ('88','32','31')
"""
res = work_conn.execute(text(sql_ejec_cgr_i))
_log("#ejec_cgr_i (faltantes CGR)", res.rowcount)


In [ ]:
# /*Ejecucion cgr del periodo, para GORES mes dipres*/
# /*determina mes max info CGR*/
work_conn.execute(text("DROP TABLE IF EXISTS #mes_max_cgr"))
sql_mes_max_cgr = f"""
SELECT AÑO, MAX(MES) AS MAX_MES
INTO #mes_max_cgr
FROM GOBGENER.dbo.EJECUCION T1
WHERE T1.AÑO = {int(Year)}
GROUP BY AÑO
"""
work_conn.execute(text(sql_mes_max_cgr))


In [ ]:
# /*data cgr año anterior mes siguiente para imputar GORE a DIPRES*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cgr_i_b"))
sql_ejec_cgr_i_b = """
SELECT T1.*
INTO #ejec_cgr_i_b
FROM GOBGENER.dbo.EJECUCION T1, #mes_max_cgr T2, GOBGENER.dbo.CR18_APERTURA T3
WHERE T1.AÑO = T2.AÑO - 1 AND T1.MES = T2.MAX_MES + 1
  AND T1.C_ENTIDAD = T3.ENTIDAD AND T3.NIVEL = 'REGIONAL'
"""
res = work_conn.execute(text(sql_ejec_cgr_i_b))
_log("#ejec_cgr_i_b (GORE mes DIPRES)", res.rowcount)


In [ ]:
# data EJEC_CGR_I; set EJEC_CGR_I EJEC_CGR_I_b; run;
# /*datos para imputar tribunales, y gores faltantes desde CGR, y también para imputar Gores para mes dipres*/
res = work_conn.execute(text("INSERT INTO #ejec_cgr_i SELECT * FROM #ejec_cgr_i_b"))
# data EJEC_CGR_I (drop=n_item n_asignacion C_SUBASIGNACION N_SUBASIGNACION PPTO_INICIAL
#                       PPTO_ACTUAL PERCIB_PAG POR_PERCIB_PAG MAX_MES MAX_MES_ENT)
work_conn.execute(text(
    "ALTER TABLE #ejec_cgr_i DROP COLUMN N_ITEM, N_ASIGNACION, C_SUBASIGNACION, "
    "N_SUBASIGNACION, PPTO_INICIAL, PPTO_ACTUAL, PERCIB_PAG, POR_PERCIB_PAG"
))
# proc sql; drop table EJEC_CGR_I_b, MES_MAX_CGR; quit;
for _t in ["#ejec_cgr_i_b", "#mes_max_cgr"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))
_log("INSERT #ejec_cgr_i <- #ejec_cgr_i_b", res.rowcount)


In [ ]:
# El SAS hace SELECT t2.* junto a un GROUP BY parcial; T-SQL no lo admite, asi que
# la lista de columnas se toma de la propia tabla temporal ya construida
cols_ejec_cgr_i = list(pd.read_sql(text("SELECT TOP 0 * FROM #ejec_cgr_i"), work_conn).columns)

# columnas de t2 que el GROUP BY del SAS lista explicitamente
grp_muestra = [
    "SUBSECTOR", "MONEDA", "MES", "AÑO", "C_PARTIDA", "N_PARTIDA", "C_CAPITULO",
    "N_CAPITULO", "C_PROGRAMA", "N_PROGRAMA", "C_TIPO_CUENTA", "C_CUENTA", "N_CUENTA",
    "C_ITEM", "C_ASIGNACION", "C_ANALITICO", "N_ANALITICO", "DEVENG",
]
_grp_muestra_up = {c.upper() for c in grp_muestra}
# las columnas de t2 fuera del GROUP BY toman el valor del primer registro del grupo
# en PROC SQL: se replican con MIN()
sel_muestra = ", ".join(
    (f"t2.{c}" if c.upper() in _grp_muestra_up else f"MIN(t2.{c}) AS {c}")
    for c in cols_ejec_cgr_i
)
grp_muestra_sql = ", ".join(f"t2.{c}" for c in grp_muestra)


In [ ]:
# /* CREA TABLA DE REDISTRIBUCION DE MUESTRA DE TRANSACCIONES MIXTAS (M) SEGUN ESTRUCTURA, PARA EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mixtas_m"))
sql_muestra_mixtas = f"""
SELECT {sel_muestra},
       t1.C_SCN, t1.C_CAGENTE, t1.C_CUP,
       SUM(t2.DEVENG / 1000000.0 * t1.POND) AS DEVENGO
INTO #ejec_muestra_mixtas_m
FROM GOBGENER.dbo.EST_MIXTAS_M t1
LEFT JOIN #ejec_cgr_i t2
       ON (t2.C_ENTIDAD = t1.ENTIDAD) AND (t2.C_ASIGNACION = t1.C_ASIGNACION)
      AND (t2.C_ANALITICO = t1.C_ANALITICO) AND (t2.C_CUENTA = t1.C_CUENTA)
      AND (t2.C_ITEM = t1.C_ITEM) AND (t2.MES = t1.MES)
WHERE t2.C_CUENTA IN ('24','33') AND t2.C_ITEM IN ('03','01','06','08','09')
  AND t1.ENTIDAD NOT IN (5601, 10201) AND t2.DEVENG IS NOT NULL
GROUP BY {grp_muestra_sql}, t1.ENTIDAD, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP
"""
res = work_conn.execute(text(sql_muestra_mixtas))
# DATA EJEC_MUESTRA_MIXTAS_M; SET EJEC_MUESTRA_MIXTAS_M (DROP=DEVENG); RUN;
work_conn.execute(text("ALTER TABLE #ejec_muestra_mixtas_m DROP COLUMN DEVENG"))
_log("#ejec_muestra_mixtas_m", res.rowcount)


In [ ]:
# /* CREA TABLA DE ANALITICOS DE MIXTAS (M) A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_ejec_exp_mixtas_m_2"))
sql_cruce_exp = """
SELECT t2.*, t1.Clasif_Tcons
INTO #cruce_ejec_exp_mixtas_m_2
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN #ejec_cgr_i t2
       ON (t2.C_ENTIDAD = t1.ENTIDAD) AND (t2.C_ASIGNACION = t1.C_ASIGNACION)
      AND (t2.C_ANALITICO = t1.C_ANALITICO) AND (t2.C_CUENTA = t1.C_CUENTA)
      AND (t2.C_ITEM = t1.C_ITEM)
WHERE t1.M_MUESTRA = 0 AND t1.ENTIDAD NOT IN (5601, 10201)
  AND t1.Clasif_Tcons = 'M' AND t2.DEVENG IS NOT NULL
"""
res = work_conn.execute(text(sql_cruce_exp))
_log("#cruce_ejec_exp_mixtas_m_2", res.rowcount)


In [ ]:
# /* CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE
#    TRANSACCIONES MIXTAS (M), PARA EJECUCION AÑO t */
cols_cruce_exp = cols_ejec_cgr_i + ["Clasif_Tcons"]
grp_prelim2 = [
    "SUBSECTOR", "MONEDA", "AÑO", "MES", "C_ENTIDAD", "C_PARTIDA", "N_PARTIDA",
    "C_CAPITULO", "N_CAPITULO", "C_PROGRAMA", "N_PROGRAMA", "C_TIPO_CUENTA",
    "C_CUENTA", "N_CUENTA", "C_ITEM", "C_ASIGNACION", "C_ANALITICO", "N_ANALITICO",
    "DEVENG", "Clasif_Tcons",
]
_grp_prelim2_up = {c.upper() for c in grp_prelim2}
sel_prelim2 = ", ".join(
    (f"t2.{c}" if c.upper() in _grp_prelim2_up else f"MIN(t2.{c}) AS {c}")
    for c in cols_cruce_exp
)
grp_prelim2_sql = ", ".join(f"t2.{c}" for c in grp_prelim2)

work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_prelim2"))
sql_prelim2 = f"""
SELECT {sel_prelim2},
       t1.C_SCN, t1.C_CUP, t1.C_CAGENTE,
       SUM(t2.DEVENG) AS DATO_OR,
       SUM(CASE WHEN t3.DATO IS NOT NULL THEN t2.DEVENG / 1000000.0 * t1.POND * t3.DATO
                ELSE t2.DEVENG / 1000000.0 * t1.POND END) AS DEVENGO
INTO #ejec_exp_mixtas_m_prelim2
FROM GOBGENER.dbo.EST_EXP_MIXTAS_M t1
LEFT JOIN #cruce_ejec_exp_mixtas_m_2 t2
       ON (t2.C_CUENTA = t1.C_CUENTA) AND (t2.C_ITEM = t1.C_ITEM)
      AND (t2.C_ASIGNACION = t1.C_ASIGNACION) AND (t2.C_ANALITICO = t1.C_ANALITICO)
      AND (t2.MES = t1.MES) AND (t2.C_ENTIDAD = t1.ENTIDAD)
LEFT JOIN #prom_mes t3
       ON (t2.MONEDA = t3.MONEDA) AND (t2.MES = t3.MES) AND (t2.AÑO = t3.AÑO)
WHERE t2.DEVENG IS NOT NULL
GROUP BY {grp_prelim2_sql}, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP
"""
res = work_conn.execute(text(sql_prelim2))
_log("#ejec_exp_mixtas_m_prelim2", res.rowcount)


In [ ]:
# DATA EJEC_EXP_MIXTAS_M_prelim; SET EJEC_EXP_MIXTAS_M_prelim2 (DROP=DEVENG DATO_OR Clasif_Tcons); RUN;
cols_mixtas_total = [c for c in cols_ejec_cgr_i if c.upper() != "DEVENG"] + [
    "C_SCN", "C_CAGENTE", "C_CUP", "DEVENGO",
]
cols_mixtas_total_sql = ", ".join(cols_mixtas_total)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_prelim"))
work_conn.execute(text(
    f"SELECT {cols_mixtas_total_sql} INTO #ejec_exp_mixtas_m_prelim FROM #ejec_exp_mixtas_m_prelim2"
))


In [ ]:
# /* Crea tabla con la ejecución total de transacciones Mixtas (M), Muestra + Resto */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mixtas_m_total"))
work_conn.execute(text(
    f"SELECT {cols_mixtas_total_sql} INTO #ejec_mixtas_m_total FROM #ejec_muestra_mixtas_m"
))
res = work_conn.execute(text(
    f"INSERT INTO #ejec_mixtas_m_total ({cols_mixtas_total_sql}) "
    f"SELECT {cols_mixtas_total_sql} FROM #ejec_exp_mixtas_m_prelim"
))
_log("INSERT #ejec_mixtas_m_total <- #ejec_exp_mixtas_m_prelim", res.rowcount)


In [ ]:
# /* Ajuste por imputación de D63 en pandemia (2020) */
sql_ajuste_pandemia = """
DELETE FROM #ejec_mixtas_m_total
WHERE C_ENTIDAD = 911 AND C_CUENTA = '24' AND C_ITEM = '03' AND C_ASIGNACION = '002'
  AND C_ANALITICO IN ('03002000') AND AÑO = 2020 AND MES > 4
"""
res = work_conn.execute(text(sql_ajuste_pandemia))
_log("DELETE ajuste D63 pandemia 2020 en #ejec_mixtas_m_total", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para ANALISIS DESAGREGADO CG coyuntura */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_g_mixt_aper"))
sql_g_mixt_aper = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       {int(Year)} AS AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS [TRIM],
       t1.MES,
       t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t1.C_SCN,
       t1.C_CAGENTE,
       SUM(t1.DEVENGO) AS Dato,
       t3.NIVEL,
       (CASE WHEN t3.ACTIVIDAD = 'EEPP' THEN 'EEPP'
             WHEN t3.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             ELSE 'AAPP' END) AS C_ACT,
       'TC' AS C_ACTIVO
INTO #ejec_g_mixt_aper
FROM #ejec_mixtas_m_total t1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t3 ON (t1.C_ENTIDAD = t3.ENTIDAD)
GROUP BY t1.SUBSECTOR,
         t1.MONEDA,
         t1.AÑO,
         (CASE WHEN t1.MES IN (1,2,3) THEN 1
               WHEN t1.MES IN (4,5,6) THEN 2
               WHEN t1.MES IN (7,8,9) THEN 3
               ELSE 4 END),
         t1.MES,
         t1.C_ENTIDAD,
         t1.C_PARTIDA,
         t1.C_CAPITULO,
         t1.C_PROGRAMA,
         t1.C_TIPO_CUENTA,
         t1.C_CUENTA,
         t1.C_ITEM,
         t1.C_ASIGNACION,
         t1.C_ANALITICO,
         t1.N_ANALITICO,
         t1.C_SCN,
         t1.C_CAGENTE,
         t3.NIVEL,
         (CASE WHEN t3.ACTIVIDAD = 'EEPP' THEN 'EEPP'
               WHEN t3.ACTIVIDAD = 'SSPP' THEN 'SSPP'
               ELSE 'AAPP' END)
"""
res = work_conn.execute(text(sql_g_mixt_aper))
_log("#ejec_g_mixt_aper", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para CG coyuntura, agregado */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_g_mixt"))
sql_g_mixt = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       t1.[TRIM],
       t1.MES,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_SCN,
       t1.C_CAGENTE,
       t1.C_ACTIVO,
       SUM(t1.Dato) AS Dato,
       t1.C_ACT,
       t1.NIVEL
INTO #ejec_g_mixt
FROM #ejec_g_mixt_aper t1
GROUP BY t1.SUBSECTOR,
         t1.MONEDA,
         t1.AÑO,
         t1.[TRIM],
         t1.MES,
         t1.C_TIPO_CUENTA,
         t1.C_CUENTA,
         t1.C_ITEM,
         t1.C_SCN,
         t1.C_CAGENTE,
         t1.C_ACT,
         t1.NIVEL,
         t1.C_ACTIVO
"""
work_conn.execute(text(sql_g_mixt))
ejec_g_mixt = pd.read_sql(text("SELECT * FROM #ejec_g_mixt"), work_conn)
_log("ejec_g_mixt", ejec_g_mixt)


In [ ]:
# PROC SQL; DROP TABLE EJEC_MUESTRA_MIXTAS_M, CRUCE_EJEC_EXP_MIXTAS_M,
#           EJEC_EXP_MIXTAS_M_prelim, EJEC_EXP_MIXTAS_M, EJEC_MIXTAS_M_TOTAL; QUIT;
for _t in ["#ejec_muestra_mixtas_m", "#ejec_exp_mixtas_m_prelim", "#ejec_mixtas_m_total"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## 2b_Salida EJEC hasta CGR

Consolida la ejecución de Contraloría del año en gasto de gobierno (remuneraciones, consumo y transferencias), valorizada con el tipo de cambio promedio mensual y clasificada por transacción SCN y actividad, con el reparto 92,8%/7,2% de las transferencias en especie de salud, y entrega la salida detallada y la agregada sin montos en cero

*confianza: medium · verificador: approve · SAS: Cadena de PROC SQL (CREATE TABLE con LEFT JOIN, UPDATE, DELETE, APPEND) + DATA step SET sobre datasets WORK*

In [ ]:
# ========= 2b_Salida EJEC hasta CGR =========
# /* Salida para indicadores GC de consumo de gobierno (coyuntura) DATA DE CONTRALORIA (t-1) */
# /*Analiticos predefinidos */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim"))
sql_a_prelim = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       CASE WHEN t1.MES IN (1,2,3) THEN 1
            WHEN t1.MES IN (4,5,6) THEN 2
            WHEN t1.MES IN (7,8,9) THEN 3
            ELSE 4 END AS TRIM,
       t1.MES,
       t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
       t1.C_ANALITICO, t1.N_ANALITICO,
       CASE WHEN t2.ANALITICO = 'M' THEN t1.N_ANALITICO ELSE t2.ANALITICO END AS ANALITICO_,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       CASE WHEN t4.ACTIVIDAD IN ('FFAA','FFOO','PREVISION','FONASA','CORFO','CENABAST','AAPP') THEN 'AAPP'
            WHEN t4.ACTIVIDAD = 'SSPP' THEN 'SSPP'
            WHEN t4.ACTIVIDAD = 'EEPP' THEN 'EEPP' END AS C_ACT,
       t4.NIVEL,
       t1.DEVENG * t3.DATO / 1000000.0 AS DEVENGO
INTO #ejecucion_a_prelim
FROM #ejec_cgr_i t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
       ON t1.C_TIPO_CUENTA = t2.T_CUENTA
      AND t1.C_CUENTA = t2.C_CUENTA
      AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
       ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
       ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t1.C_CUENTA NOT IN ('05','13','24','33')
  AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.C_ENTIDAD NOT IN (5601, 10201)
"""
res = work_conn.execute(text(sql_a_prelim))
_log("#ejecucion_a_prelim", res.rowcount)


In [ ]:
# /*Este Ajuste lo pidieron de síntesis, incorpora al indicador de transferencias en esp
#   lo que consideran desde anuales, el porcentaje sale de anuales...*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_aj"))
sql_a_prelim_aj = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES,
       t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
       t1.C_ANALITICO, t1.N_ANALITICO, t1.ANALITICO_,
       t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
       CASE WHEN t1.C_CUENTA = '22' AND SUBSTRING(t1.C_ANALITICO,1,5) = '12999'
                 AND t1.C_PARTIDA = '16' AND t1.C_ACT = 'SSPP'
            THEN t1.DEVENGO * 0.928
            ELSE t1.DEVENGO END AS DEVENGO
INTO #ejecucion_a_prelim_aj
FROM #ejecucion_a_prelim t1
"""
res = work_conn.execute(text(sql_a_prelim_aj))
_log("#ejecucion_a_prelim_aj", res.rowcount)


In [ ]:
sql_upd_aj = """
UPDATE #ejecucion_a_prelim_aj
SET C_SCN = 'D63',
    C_CAGENTE = 'S14',
    C_TRANSACCION = '433',
    T_TRANSACCION = 'transfesp',
    N_CUENTA = 'IG',
    C_CUP = '272'
WHERE C_CUENTA = '22' AND SUBSTRING(C_ANALITICO,1,5) = '12999'
  AND C_PARTIDA = '16' AND C_ACT = 'SSPP'
"""
res = work_conn.execute(text(sql_upd_aj))
_log("UPDATE #ejecucion_a_prelim_aj (reclasificación D63)", res.rowcount)


In [ ]:
# CREATE TABLE EJECUCION_A_prelim AS SELECT ... DEVENGO*(1-0.928) ... -> el SAS recrea la
# tabla copiando todas las columnas idénticas y solo cambia DEVENGO en las filas SSPP:
# equivale a un UPDATE de ese campo sobre las mismas filas
sql_upd_prelim = """
UPDATE #ejecucion_a_prelim
SET DEVENGO = DEVENGO * (1 - 0.928)
WHERE C_CUENTA = '22' AND SUBSTRING(C_ANALITICO,1,5) = '12999'
  AND C_PARTIDA = '16' AND C_ACT = 'SSPP'
"""
res = work_conn.execute(text(sql_upd_prelim))
_log("UPDATE #ejecucion_a_prelim (7,2% remanente)", res.rowcount)


In [ ]:
# PROC DATASETS APPEND BASE=EJECUCION_A_prelim DATA=EJECUCION_A_prelim_Aj FORCE
# El SAS recrea antes la tabla Aj dejando solo las filas SSPP: ese filtro va en el SELECT
cols_prelim = (
    "SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, "
    "C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, ANALITICO_, "
    "C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, DEVENGO"
)
sql_append_aj = f"""
INSERT INTO #ejecucion_a_prelim ({cols_prelim})
SELECT {cols_prelim}
FROM #ejecucion_a_prelim_aj
WHERE C_CUENTA = '22' AND SUBSTRING(C_ANALITICO,1,5) = '12999'
  AND C_PARTIDA = '16' AND C_ACT = 'SSPP'
"""
res = work_conn.execute(text(sql_append_aj))
_log("APPEND #ejecucion_a_prelim <- #ejecucion_a_prelim_aj", res.rowcount)


In [ ]:
sql_del_32 = """
DELETE FROM #ejecucion_a_prelim
WHERE C_CUENTA = '32' AND C_ANALITICO IN ('04001000','04002000')
"""
res = work_conn.execute(text(sql_del_32))
_log("DELETE #ejecucion_a_prelim (C_CUENTA 32 analíticos variables)", res.rowcount)


In [ ]:
# /*Analiticos predefinidos: AJUSTE COYUNTURA - se elimina de indicador transacción CORFO
#   que se desagrega anualmente entre ventas y D45 */
sql_del_corfo = """
DELETE FROM #ejecucion_a_prelim
WHERE C_ENTIDAD = 3801 AND C_CUENTA = '06' AND C_ITEM = '01'
  AND C_ASIGNACION = '000' AND C_ANALITICO IN ('01000000')
"""
res = work_conn.execute(text(sql_del_corfo))
_log("DELETE #ejecucion_a_prelim (CORFO)", res.rowcount)


In [ ]:
# /*Analiticos predefinidos (sigue)*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a"))
sql_a = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES,
       t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
       t1.C_ANALITICO, t1.N_ANALITICO,
       t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
       SUM(t1.DEVENGO) AS DATO
INTO #ejecucion_a
FROM #ejecucion_a_prelim t1
WHERE t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES,
         t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
         t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
         t1.C_ANALITICO, t1.N_ANALITICO,
         t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
         t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL
"""
res = work_conn.execute(text(sql_a))
_log("#ejecucion_a", res.rowcount)


In [ ]:
# /*Analiticos variables */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b"))
sql_b = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       CASE WHEN t1.MES IN (1,2,3) THEN 1
            WHEN t1.MES IN (4,5,6) THEN 2
            WHEN t1.MES IN (7,8,9) THEN 3
            ELSE 4 END AS TRIM,
       t1.MES,
       t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
       t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       CASE WHEN t4.ACTIVIDAD IN ('FFAA','FFOO','PREVISION','FONASA','CORFO','CENABAST','AAPP') THEN 'AAPP'
            WHEN t4.ACTIVIDAD = 'SSPP' THEN 'SSPP'
            WHEN t4.ACTIVIDAD = 'EEPP' THEN 'EEPP' END AS C_ACT,
       t4.NIVEL,
       CASE WHEN t3.DATO IS NOT NULL THEN t1.DEVENG * t3.DATO / 1000000.0
            ELSE t1.DEVENG / 1000000.0 END AS DATO
INTO #ejecucion_b
FROM #ejec_cgr_i t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
       ON t1.C_TIPO_CUENTA = t2.T_CUENTA
      AND t1.C_CUENTA = t2.C_CUENTA
      AND t1.C_ITEM = t2.C_ITEM
      AND t1.C_ASIGNACION = t2.C_ASIGNACION
      AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
       ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
       ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t1.C_CUENTA = '32'
  AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.C_ANALITICO IN ('04001000','04002000')
  AND t1.C_ENTIDAD NOT IN (5601, 10201)
"""
res = work_conn.execute(text(sql_b))
_log("#ejecucion_b", res.rowcount)


In [ ]:
# /*analíticos variables*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c"))
sql_c = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       CASE WHEN t1.MES IN (1,2,3) THEN 1
            WHEN t1.MES IN (4,5,6) THEN 2
            WHEN t1.MES IN (7,8,9) THEN 3
            ELSE 4 END AS TRIM,
       t1.MES,
       t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
       t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       CASE WHEN t4.ACTIVIDAD IN ('FFAA','FFOO','PREVISION','FONASA','CORFO','CENABAST','AAPP') THEN 'AAPP'
            WHEN t4.ACTIVIDAD = 'SSPP' THEN 'SSPP'
            WHEN t4.ACTIVIDAD = 'EEPP' THEN 'EEPP' END AS C_ACT,
       t4.NIVEL,
       t1.DEVENG * t3.DATO / 1000000.0 AS DATO
INTO #ejecucion_c
FROM #ejec_cgr_i t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
       ON t1.C_PARTIDA = t2.C_PARTIDA
      AND t1.C_CAPITULO = t2.C_CAPITULO
      AND t1.C_PROGRAMA = t2.C_PROGRAMA
      AND t1.C_ENTIDAD = t2.ENTIDAD
      AND t1.C_TIPO_CUENTA = t2.T_CUENTA
      AND t1.C_CUENTA = t2.C_CUENTA
      AND t1.C_ITEM = t2.C_ITEM
      AND t1.C_ASIGNACION = t2.C_ASIGNACION
      AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
       ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
       ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t1.C_CUENTA IN ('05','13','24','33')
  AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND (t2.OBS IS NULL OR t2.OBS NOT IN ('CR18_difcoy','CR18_difact'))
  AND t1.C_ENTIDAD NOT IN (5601, 10201)
  AND t2.C_SCN NOT IN ('TC')
"""
res = work_conn.execute(text(sql_c))
_log("#ejecucion_c", res.rowcount)


In [ ]:
# /* Transacciones diferenciadas para coyuntura */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d"))
sql_d = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       CASE WHEN t1.MES IN (1,2,3) THEN 1
            WHEN t1.MES IN (4,5,6) THEN 2
            WHEN t1.MES IN (7,8,9) THEN 3
            ELSE 4 END AS TRIM,
       t1.MES,
       t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
       t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_ACTIVO AS C_SCN,
       t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       CASE WHEN t4.ACTIVIDAD IN ('FFAA','FFOO','PREVISION','FONASA','CORFO','CENABAST','AAPP') THEN 'AAPP'
            WHEN t4.ACTIVIDAD = 'SSPP' THEN 'SSPP'
            WHEN t4.ACTIVIDAD = 'EEPP' THEN 'EEPP' END AS C_ACT,
       t4.NIVEL,
       t1.DEVENG * t3.DATO / 1000000.0 AS DATO
INTO #ejecucion_d
FROM #ejec_cgr_i t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
       ON t1.C_PARTIDA = t2.C_PARTIDA
      AND t1.C_CAPITULO = t2.C_CAPITULO
      AND t1.C_PROGRAMA = t2.C_PROGRAMA
      AND t1.C_ENTIDAD = t2.ENTIDAD
      AND t1.C_TIPO_CUENTA = t2.T_CUENTA
      AND t1.C_CUENTA = t2.C_CUENTA
      AND t1.C_ITEM = t2.C_ITEM
      AND t1.C_ASIGNACION = t2.C_ASIGNACION
      AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
       ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
       ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t2.OBS IN ('CR18_difcoy')
  AND t1.C_ENTIDAD NOT IN (5601, 10201)
  AND t2.C_ACTIVO IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND (t2.C_SCN IS NULL OR t2.C_SCN NOT IN ('TC'))
"""
res = work_conn.execute(text(sql_d))
_log("#ejecucion_d", res.rowcount)


In [ ]:
# /* Transacciones diferenciadas por actividad */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f"))
sql_f = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       CASE WHEN t1.MES IN (1,2,3) THEN 1
            WHEN t1.MES IN (4,5,6) THEN 2
            WHEN t1.MES IN (7,8,9) THEN 3
            ELSE 4 END AS TRIM,
       t1.MES,
       t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
       t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       t2.C_CUP AS C_ACT,
       t4.NIVEL,
       t1.DEVENG * t3.DATO / 1000000.0 AS DATO
INTO #ejecucion_f
FROM #ejec_cgr_i t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
       ON t1.C_PARTIDA = t2.C_PARTIDA
      AND t1.C_CAPITULO = t2.C_CAPITULO
      AND t1.C_PROGRAMA = t2.C_PROGRAMA
      AND t1.C_ENTIDAD = t2.ENTIDAD
      AND t1.C_TIPO_CUENTA = t2.T_CUENTA
      AND t1.C_CUENTA = t2.C_CUENTA
      AND t1.C_ITEM = t2.C_ITEM
      AND t1.C_ASIGNACION = t2.C_ASIGNACION
      AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
       ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
       ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t2.OBS IN ('CR18_difact')
  AND t1.C_ENTIDAD NOT IN (5601, 10201)
  AND t2.C_SCN NOT IN ('TC')
  AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
"""
res = work_conn.execute(text(sql_f))
_log("#ejecucion_f", res.rowcount)


In [ ]:
# /* Transacciones TCONS (sólo aplica Mixtas) EJEC_G_MIXT_aper*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_g"))
sql_g = """
SELECT *
INTO #ejecucion_g
FROM #ejec_g_mixt_aper
WHERE C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
"""
res = work_conn.execute(text(sql_g))
_log("#ejecucion_g", res.rowcount)


In [ ]:
# /*UNE TABLAS de Salida total de ejecución CG coy de data CGR*/
cols_ejec = (
    "SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, "
    "C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, "
    "C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, DATO"
)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_cgr_prev_2"))
sql_prev_2 = f"""
SELECT {cols_ejec} INTO #ejecucion_cg_cgr_prev_2 FROM #ejecucion_a
UNION ALL SELECT {cols_ejec} FROM #ejecucion_b
UNION ALL SELECT {cols_ejec} FROM #ejecucion_c
UNION ALL SELECT {cols_ejec} FROM #ejecucion_d
UNION ALL SELECT {cols_ejec} FROM #ejecucion_g
UNION ALL SELECT {cols_ejec} FROM #ejecucion_f
"""
res = work_conn.execute(text(sql_prev_2))
_log("#ejecucion_cg_cgr_prev_2", res.rowcount)


In [ ]:
# DROP TABLE EJECUCION_A_prelim, EJECUCION_A, ... EJECUCION_A_PRELIM_AJ
for _t in [
    "#ejecucion_a_prelim",
    "#ejecucion_a",
    "#ejecucion_b",
    "#ejecucion_c",
    "#ejecucion_d",
    "#ejecucion_g",
    "#ejecucion_f",
    "#ejecucion_a_prelim_aj",
]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


In [ ]:
# /*SALIDA DETALLADA CGR - IMPUTACIONES*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_cgr_aper_imp"))
sql_aper_imp = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       {int(Year)} AS AÑO,
       t1.TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
       t1.C_ANALITICO, t1.N_ANALITICO,
       t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
       SUM(t1.DATO) AS DATO,
       'Imputación cobertura' AS OBS
INTO #ejecucion_cg_cgr_aper_imp
FROM #ejecucion_cg_cgr_prev_2 t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD,
         t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA,
         t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
         t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
         t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL
"""
res = work_conn.execute(text(sql_aper_imp))
_log("#ejecucion_cg_cgr_aper_imp", res.rowcount)


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_cg_cgr_aper_imp WHERE DATO = 0"))
_log("DELETE #ejecucion_cg_cgr_aper_imp (dato=0)", res.rowcount)


In [ ]:
# /*SALIDA AGREGADA CGR - IMPUTACIONES*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_cgr_imp"))
sql_imp = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       {int(Year)} AS AÑO,
       t1.TRIM,
       t1.MES,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
       SUM(t1.DATO) AS DATO,
       'Imputación cobertura' AS OBS
INTO #ejecucion_cg_cgr_imp
FROM #ejecucion_cg_cgr_prev_2 t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES,
         t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
         t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
         t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL
"""
res = work_conn.execute(text(sql_imp))
_log("#ejecucion_cg_cgr_imp", res.rowcount)


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_cg_cgr_imp WHERE DATO = 0"))
_log("DELETE #ejecucion_cg_cgr_imp (dato=0)", res.rowcount)


In [ ]:
# drop table EJECUCION_CG_CGR_PREV_2
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_cgr_prev_2"))


## 3a_Imputa_Mixtas_Faltantes

Ejecución mensual de transacciones mixtas imputada con datos DIPRES en los periodos que Contraloría aún no informa, sumando la muestra y su expansión al resto de entidades

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE (LEFT JOIN + GROUP BY + CASE) y DATA step SET/DROP*

In [ ]:
# ========= 3a_Imputa_Mixtas_Faltantes =========
# /*II. IMPUTA FALTANTES EN INFO CONTRALORIA USANDO DATOS DIPRES*/
work_conn.execute(text("DROP TABLE IF EXISTS #entid_dip_y1"))
sql_entid_dip_y1 = f"""
SELECT DISTINCT *
INTO #entid_dip_y1
FROM GOBGENER.dbo.CR18_ENTID_DIP_Y
WHERE AÑO = {int(Year) - 1}
"""
res = work_conn.execute(text(sql_entid_dip_y1))
_log("#entid_dip_y1", res.rowcount)


In [ ]:
# COBERTURA_DETALLE y PROM_MES llegan en memoria desde nodos anteriores: se
# materializan en la sesión para poder cruzarlos en SQL con las tablas de BD
cobertura_detalle.to_sql("#cobertura_detalle", work_conn, if_exists="replace", index=False)
prom_mes.to_sql("#prom_mes", work_conn, if_exists="replace", index=False)
_log("#cobertura_detalle", cobertura_detalle)


In [ ]:
# Helpers de armado de SQL para este nodo: PROC SQL admite SELECT t.* junto a
# un GROUP BY (remerge) y el SET de un DATA step admite tablas con distintas
# columnas; T-SQL no hace ni lo uno ni lo otro.
def _sel_grupo(cols, group_cols, alias):
    """SELECT alias.* con GROUP BY: las columnas agrupadas van tal cual y el
    resto se remergea con el valor del primer registro del grupo (MIN)."""
    grupo = {c.upper() for c in group_cols}
    return ", ".join(
        f"{alias}.[{c}]" if c.upper() in grupo else f"MIN({alias}.[{c}]) AS [{c}]"
        for c in cols
    )


def _sel_union(cols_total, cols_tabla):
    """SET a b: alinea por nombre y completa con NULL lo que la tabla no trae."""
    presentes = {c.upper() for c in cols_tabla}
    return ", ".join(
        f"[{c}]" if c.upper() in presentes else f"NULL AS [{c}]" for c in cols_total
    )


In [ ]:
# /*TRABAJA TCONS MIXTAS DIPRES*/
# /*Ejecucion dipres del periodo, con entidad asignada*/
# El SELECT T1.* y el DROP= del DATA step siguiente (PPTO_INICIAL PPTO_ACTUAL
# EJEC_ACUM MAX_MES MAX_MES_ENT) se resuelven como lista de columnas.
_drop_ejec_dip = {"PPTO_INICIAL", "PPTO_ACTUAL", "EJEC_ACUM", "MAX_MES", "MAX_MES_ENT", "C_ENTIDAD", "ACTIVIDAD", "NIVEL"}
_cols_dipres = list(pd.read_sql(text("SELECT TOP 0 * FROM GOBGENER.dbo.EJEC_DIPRES"), engine).columns)
sel_t1_dipres = ", ".join(f"t1.[{c}]" for c in _cols_dipres if c.upper() not in _drop_ejec_dip)

work_conn.execute(text("DROP TABLE IF EXISTS #ejec_dip_i"))
sql_ejec_dip_i = f"""
SELECT {sel_t1_dipres},
       CASE WHEN t2.C_ENTIDAD IS NULL AND t4.C_ENTIDAD IS NOT NULL THEN t4.C_ENTIDAD
            WHEN t4.C_ENTIDAD IS NULL AND t2.C_ENTIDAD IS NULL
                 THEN TRY_CAST(CONCAT(LTRIM(RTRIM(t1.C_PARTIDA)), LTRIM(RTRIM(t1.C_CAPITULO)), LTRIM(RTRIM(t1.C_PROGRAMA))) AS float)
            ELSE t2.C_ENTIDAD END AS C_ENTIDAD,
       CASE WHEN (t2.ACTIVIDAD IS NULL OR t2.ACTIVIDAD = '')
                 AND (t4.ACTIVIDAD IS NOT NULL AND t4.ACTIVIDAD <> '') THEN t4.ACTIVIDAD
            ELSE t2.ACTIVIDAD END AS ACTIVIDAD,
       CASE WHEN (t2.NIVEL IS NULL OR t2.NIVEL = '')
                 AND (t4.NIVEL IS NOT NULL AND t4.NIVEL <> '') THEN t4.NIVEL
            ELSE t2.NIVEL END AS NIVEL
INTO #ejec_dip_i
FROM GOBGENER.dbo.EJEC_DIPRES t1
LEFT JOIN GOBGENER.dbo.CR18_ENTID_DIP_Y t2
       ON t1.AÑO = t2.AÑO AND t1.C_PARTIDA = t2.C_PARTIDA
      AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
LEFT JOIN #entid_dip_y1 t4
       ON t1.C_PARTIDA = t4.C_PARTIDA AND t1.C_CAPITULO = t4.C_CAPITULO
      AND t1.C_PROGRAMA = t4.C_PROGRAMA
LEFT JOIN #cobertura_detalle t3
       ON t1.AÑO = t3.AÑO AND t1.C_PARTIDA = t3.C_PARTIDA
      AND t1.C_CAPITULO = t3.C_CAPITULO AND t1.C_PROGRAMA = t3.C_PROGRAMA
      AND t1.MONEDA = t3.MONEDA
WHERE t1.AÑO = {int(Year)}
  AND t3.ID = 'Revisar'
  AND t1.MES <= t3.MAX_MES
  AND t1.MES > t3.MAX_MES_ENT
  AND t1.C_PARTIDA NOT IN ('88', '62', '31')
"""
res = work_conn.execute(text(sql_ejec_dip_i))
_log("#ejec_dip_i", res.rowcount)


In [ ]:
# /*TCONS MIXTAS DE LA MUESTRA DIPRES*/
cols_ejec_dip_i = list(pd.read_sql(text("SELECT TOP 0 * FROM #ejec_dip_i"), work_conn).columns)
_up_dip_i = {c.upper() for c in cols_ejec_dip_i}
grp_muestra = [
    c for c in [
        "SUBSECTOR", "MONEDA", "MES", "AÑO", "C_PARTIDA", "C_CAPITULO", "C_PROGRAMA",
        "C_TIPO_CUENTA", "C_CUENTA", "C_ITEM", "C_ASIGNACION", "C_ANALITICO",
        "N_ANALITICO", "C_ENTIDAD", "ACTIVIDAD", "NIVEL", "DEVENG",
    ] if c in _up_dip_i
]
sel_muestra_t2 = _sel_grupo(cols_ejec_dip_i, grp_muestra, "t2")
grp_muestra_sql = ", ".join(f"t2.[{c}]" for c in grp_muestra)

work_conn.execute(text("DROP TABLE IF EXISTS #muestra_mixtas_m_prev_dip"))
sql_muestra_prev = f"""
SELECT {sel_muestra_t2},
       t1.C_SCN,
       t1.C_CAGENTE,
       CAST(t1.C_CUP AS char(4)) AS C_CUP,
       SUM(t2.DEVENG / 1000000.0 * t1.POND) AS DEVENGO
INTO #muestra_mixtas_m_prev_dip
FROM GOBGENER.dbo.EST_MIXTAS_M t1
LEFT JOIN #ejec_dip_i t2
       ON t2.C_ENTIDAD = t1.ENTIDAD AND t2.C_ASIGNACION = t1.C_ASIGNACION
      AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.C_CUENTA = t1.C_CUENTA
      AND t2.C_ITEM = t1.C_ITEM AND t2.MES = t1.MES
WHERE t2.C_CUENTA IN ('24', '33')
  AND t2.C_ITEM IN ('03', '01', '06', '08', '09')
  AND t1.ENTIDAD NOT IN (5601, 10201)
GROUP BY {grp_muestra_sql}, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP
"""
res = work_conn.execute(text(sql_muestra_prev))
_log("#muestra_mixtas_m_prev_dip", res.rowcount)


In [ ]:
# DATA EJEC_MUESTRA_MIXTAS_M_DIP_I (DROP= DEVENG)
cols_muestra_prev = list(pd.read_sql(text("SELECT TOP 0 * FROM #muestra_mixtas_m_prev_dip"), work_conn).columns)
sel_muestra_dip_i = ", ".join(f"[{c}]" for c in cols_muestra_prev if c.upper() != "DEVENG")

work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mixtas_m_dip_i"))
sql_muestra_dip_i = f"""
SELECT {sel_muestra_dip_i}
INTO #ejec_muestra_mixtas_m_dip_i
FROM #muestra_mixtas_m_prev_dip
"""
res = work_conn.execute(text(sql_muestra_dip_i))
_log("#ejec_muestra_mixtas_m_dip_i", res.rowcount)


In [ ]:
# /* CREA TABLA DE ANALITICOS DE MIXTAS (M) A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t */
# t1.C_CUENTA/C_ITEM/C_ASIGNACION/C_ANALITICO ya vienen de t2: PROC SQL conserva
# la primera ocurrencia, así que solo se agregan ENTIDAD y Clasif_Tcons
sel_cruce_t2 = ", ".join(f"t2.[{c}]" for c in cols_ejec_dip_i)

work_conn.execute(text("DROP TABLE IF EXISTS #cruce_ejec_exp_mixtas_m_dip"))
sql_cruce = f"""
SELECT {sel_cruce_t2},
       t1.ENTIDAD,
       t1.Clasif_Tcons
INTO #cruce_ejec_exp_mixtas_m_dip
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN #ejec_dip_i t2
       ON t2.C_ENTIDAD = t1.ENTIDAD AND t2.C_ASIGNACION = t1.C_ASIGNACION
      AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.C_CUENTA = t1.C_CUENTA
      AND t2.C_ITEM = t1.C_ITEM
WHERE t1.M_MUESTRA = 0
  AND t1.ENTIDAD NOT IN (5601, 10201)
  AND t1.Clasif_Tcons = 'M'
  AND t2.DEVENG IS NOT NULL
"""
res = work_conn.execute(text(sql_cruce))
_log("#cruce_ejec_exp_mixtas_m_dip", res.rowcount)


In [ ]:
# /* CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE TRANSACCIONES MIXTAS (M),
# PARA EJECUCION AÑO t */
cols_cruce = list(pd.read_sql(text("SELECT TOP 0 * FROM #cruce_ejec_exp_mixtas_m_dip"), work_conn).columns)
_up_cruce = {c.upper() for c in cols_cruce}
grp_exp = [
    c for c in [
        "SUBSECTOR", "MONEDA", "AÑO", "MES", "C_PARTIDA", "C_CAPITULO", "C_PROGRAMA",
        "C_TIPO_CUENTA", "C_CUENTA", "C_ITEM", "C_ASIGNACION", "C_ANALITICO",
        "N_ANALITICO", "C_ENTIDAD", "ACTIVIDAD", "NIVEL", "ENTIDAD", "CLASIF_TCONS",
        "DEVENG",
    ] if c in _up_cruce
]
sel_exp_t2 = _sel_grupo(cols_cruce, grp_exp, "t2")
grp_exp_sql = ", ".join(f"t2.[{c}]" for c in grp_exp)

work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_prelim_dip"))
sql_exp_prelim = f"""
SELECT {sel_exp_t2},
       t1.C_SCN,
       t1.C_CUP,
       t1.C_CAGENTE,
       SUM(t2.DEVENG) AS DATO_OR,
       SUM(t2.DEVENG / 1000000.0 * t1.POND * t3.DATO) AS DEVENGO
INTO #ejec_exp_mixtas_m_prelim_dip
FROM GOBGENER.dbo.EST_EXP_MIXTAS_M t1
LEFT JOIN #cruce_ejec_exp_mixtas_m_dip t2
       ON t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM
      AND t2.C_ASIGNACION = t1.C_ASIGNACION AND t2.C_ANALITICO = t1.C_ANALITICO
      AND t2.MES = t1.MES AND t2.ENTIDAD = t1.ENTIDAD
LEFT JOIN #prom_mes t3
       ON t2.MONEDA = t3.MONEDA AND t2.MES = t3.MES AND t2.AÑO = t3.AÑO
WHERE t2.DEVENG IS NOT NULL
GROUP BY {grp_exp_sql}, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP
"""
res = work_conn.execute(text(sql_exp_prelim))
_log("#ejec_exp_mixtas_m_prelim_dip", res.rowcount)


In [ ]:
# DATA EJEC_EXP_MIXTAS_M_DIP_I (DROP= DEVENG DATO_OR Clasif_Tcons ENTIDAD)
cols_exp_prelim = list(pd.read_sql(text("SELECT TOP 0 * FROM #ejec_exp_mixtas_m_prelim_dip"), work_conn).columns)
_drop_exp = {"DEVENG", "DATO_OR", "CLASIF_TCONS", "ENTIDAD"}
sel_exp_dip_i = ", ".join(f"[{c}]" for c in cols_exp_prelim if c.upper() not in _drop_exp)

work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_dip_i"))
sql_exp_dip_i = f"""
SELECT {sel_exp_dip_i}
INTO #ejec_exp_mixtas_m_dip_i
FROM #ejec_exp_mixtas_m_prelim_dip
"""
res = work_conn.execute(text(sql_exp_dip_i))
_log("#ejec_exp_mixtas_m_dip_i", res.rowcount)


In [ ]:
# /* Crea tabla con la ejecución total de transacciones Mixtas (M), Muestra + Resto */
cols_muestra_dip_i = list(pd.read_sql(text("SELECT TOP 0 * FROM #ejec_muestra_mixtas_m_dip_i"), work_conn).columns)
cols_exp_dip_i = list(pd.read_sql(text("SELECT TOP 0 * FROM #ejec_exp_mixtas_m_dip_i"), work_conn).columns)
_up_muestra = {c.upper() for c in cols_muestra_dip_i}
cols_total = cols_muestra_dip_i + [c for c in cols_exp_dip_i if c.upper() not in _up_muestra]
sel_union_muestra = _sel_union(cols_total, cols_muestra_dip_i)
sel_union_exp = _sel_union(cols_total, cols_exp_dip_i)

work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mixtas_m_total_dip_imp"))
sql_total = f"""
SELECT *
INTO #ejec_mixtas_m_total_dip_imp
FROM (
    SELECT {sel_union_muestra} FROM #ejec_muestra_mixtas_m_dip_i
    UNION ALL
    SELECT {sel_union_exp} FROM #ejec_exp_mixtas_m_dip_i
) u
"""
res = work_conn.execute(text(sql_total))
_log("#ejec_mixtas_m_total_dip_imp", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para CG coyuntura */
# CALCULATED TRIM: T-SQL no admite el alias en el GROUP BY, se repite el CASE
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_g_mixt_dip_imp"))
sql_g_mixt = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       CASE WHEN t1.MES IN (1, 2, 3) THEN 1
            WHEN t1.MES IN (4, 5, 6) THEN 2
            WHEN t1.MES IN (7, 8, 9) THEN 3
            ELSE 4 END AS [TRIM],
       t1.MES,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_SCN,
       t1.C_CAGENTE,
       t1.C_CUP,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       SUM(t1.DEVENGO) AS Dato,
       t1.ACTIVIDAD,
       t1.NIVEL
INTO #ejec_g_mixt_dip_imp
FROM #ejec_mixtas_m_total_dip_imp t1
LEFT JOIN GOBGENER.dbo.MOLDE_T_SCN_2 t2
       ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
      AND t1.C_ANALITICO = t2.C_ANALITICO
WHERE t1.C_SCN IN ('D11', 'P2', 'D63')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
         CASE WHEN t1.MES IN (1, 2, 3) THEN 1
              WHEN t1.MES IN (4, 5, 6) THEN 2
              WHEN t1.MES IN (7, 8, 9) THEN 3
              ELSE 4 END,
         t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE,
         t1.C_CUP, t1.ACTIVIDAD, t1.NIVEL,
         t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP
"""
work_conn.execute(text(sql_g_mixt))
ejec_g_mixt_dip_imp = pd.read_sql(text("SELECT * FROM #ejec_g_mixt_dip_imp"), work_conn)
_log("ejec_g_mixt_dip_imp", ejec_g_mixt_dip_imp)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para ANALISIS DESAGREGADO
# CG coyuntura */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_g_mixt_aper_dip_imp"))
sql_g_mixt_aper = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       CASE WHEN t1.MES IN (1, 2, 3) THEN 1
            WHEN t1.MES IN (4, 5, 6) THEN 2
            WHEN t1.MES IN (7, 8, 9) THEN 3
            ELSE 4 END AS [TRIM],
       t1.MES,
       t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t1.C_SCN,
       t1.C_CAGENTE,
       t1.C_CUP,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       SUM(t1.DEVENGO) AS Dato,
       t1.ACTIVIDAD,
       t1.NIVEL
INTO #ejec_g_mixt_aper_dip_imp
FROM #ejec_mixtas_m_total_dip_imp t1
LEFT JOIN GOBGENER.dbo.MOLDE_T_SCN_2 t2
       ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
      AND t1.C_ANALITICO = t2.C_ANALITICO
WHERE t1.C_SCN IN ('D11', 'P2', 'D63')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
         CASE WHEN t1.MES IN (1, 2, 3) THEN 1
              WHEN t1.MES IN (4, 5, 6) THEN 2
              WHEN t1.MES IN (7, 8, 9) THEN 3
              ELSE 4 END,
         t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
         t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
         t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP, t1.ACTIVIDAD, t1.NIVEL,
         t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP
"""
work_conn.execute(text(sql_g_mixt_aper))
ejec_g_mixt_aper_dip_imp = pd.read_sql(text("SELECT * FROM #ejec_g_mixt_aper_dip_imp"), work_conn)
_log("ejec_g_mixt_aper_dip_imp", ejec_g_mixt_aper_dip_imp)


## 3b_Imputa_Salida_Coy

Arma la salida de coyuntura del gobierno central con datos DIPRES: clasifica cada devengo mensual en transacciones de cuentas nacionales, abre las transferencias en especie de la salud pública, suma las imputaciones permanentes del año y la imputación de ingresos operacionales de las entidades que solo informan el agregado, y entrega el resultado agrupado sin montos en cero ni entidades excluidas

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE / DELETE / UPDATE + DATA step SET (concatenación) + PROC DATASETS APPEND FORCE*

In [ ]:
# ========= 3b_Imputa_Salida_Coy =========
# Fragmentos SQL que el SAS repite en todos los bloques del nodo (mismo criterio)
case_trim = """(CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END)"""
case_c_act = """(CASE WHEN t1.ACTIVIDAD='FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD='EEPP' THEN 'EEPP' END)"""
scn_coy = "'D1','D11','D12','P2','P3','P72','D63','P1','P11'"
sel_ident_t1 = f"""t1.SUBSECTOR, t1.MONEDA, t1.AÑO, {case_trim} AS TRIM, t1.MES, t1.C_ENTIDAD,
       t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO"""
join_scn2 = """LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
       ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO
      AND t1.C_PROGRAMA = t2.C_PROGRAMA AND t1.C_TIPO_CUENTA = t2.T_CUENTA
      AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
      AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
      AND t1.C_ENTIDAD = t2.ENTIDAD"""
join_prom_mes = """LEFT JOIN #prom_mes t3
       ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO"""
cols_salida_list = [
    "SUBSECTOR", "MONEDA", "AÑO", "TRIM", "MES", "C_ENTIDAD", "C_PARTIDA", "C_CAPITULO",
    "C_PROGRAMA", "C_TIPO_CUENTA", "C_CUENTA", "C_ITEM", "C_ASIGNACION", "C_ANALITICO",
    "N_ANALITICO", "C_SCN", "C_CAGENTE", "C_TRANSACCION", "T_TRANSACCION", "N_CUENTA",
    "C_ACTIVO", "C_CUP", "C_ACT", "NIVEL",
]
cols_salida = ", ".join(cols_salida_list)
cols_prelim = ", ".join(cols_salida_list + ["DEVENGO"])
cols_union = ", ".join(cols_salida_list + ["Dato"])
cols_salida_t1 = ", ".join(f"t1.{c}" for c in cols_salida_list)
cols_prelim_t1 = ", ".join(f"t1.{c}" for c in cols_salida_list + ["DEVENGO"])


In [ ]:
# /*Analiticos predefinidos */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_a_prelim_dip_aper_0"))
sql_a_prelim = f"""
SELECT {sel_ident_t1},
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       {case_c_act} AS C_ACT, t1.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000) AS DEVENGO
INTO #ejec_a_prelim_dip_aper_0
FROM #ejec_dip_i t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
       ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
      AND t1.C_ANALITICO = t2.C_ANALITICO
{join_prom_mes}
WHERE t1.C_CUENTA NOT IN ('05','13','24','33')
  AND t2.C_SCN IN ({scn_coy})
  AND t1.C_ENTIDAD NOT IN (5601,10201)
"""
res = work_conn.execute(text(sql_a_prelim))
_log("#ejec_a_prelim_dip_aper_0 (EJECUCION_A_prelim_DIP_aper)", res.rowcount)


In [ ]:
# /*nuevo, para eliminar ingresos operacionales que vienen agregados en data dipres. Se incorporan al final del proceso*/
res_item = work_conn.execute(text(
    "DELETE FROM #ejec_a_prelim_dip_aper_0 WHERE C_CUENTA = '07' AND C_ITEM IN ('0','00')"
))
# C_ASIGNACION faltante en SAS es blanco y también entraba en el NOT IN: en SQL Server el NULL hay que nombrarlo
res_asig = work_conn.execute(text(
    "DELETE FROM #ejec_a_prelim_dip_aper_0 "
    "WHERE C_CUENTA = '07' AND (C_ASIGNACION IS NULL OR C_ASIGNACION NOT IN ('000'))"
))
_log("DELETE ingresos operacionales agregados (cuenta 07)",
     (res_item.rowcount or 0) + (res_asig.rowcount or 0))


In [ ]:
# /*Estructura para abrir transferencias en especie...*/
work_conn.execute(text("DROP TABLE IF EXISTS #ratios_2"))
sql_ratios_2 = """
SELECT SUBSECTOR, AÑO, MONEDA, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA,
       C_TIPO_CUENTA, C_CUENTA, C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION,
       N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, Ratio
INTO #ratios_2
FROM #ratios
WHERE C_SCN = 'D63'
"""
res = work_conn.execute(text(sql_ratios_2))
_log("#ratios_2", res.rowcount)


In [ ]:
# /*apertura transferencias en especie de la salud pública*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_a_prelim_dip_aper_r"))
sql_prelim_ratio = f"""
SELECT {cols_prelim_t1}, t2.Ratio
INTO #ejec_a_prelim_dip_aper_r
FROM #ejec_a_prelim_dip_aper_0 t1
LEFT JOIN #ratios_2 t2
       ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO
      AND t1.C_PROGRAMA = t2.C_PROGRAMA AND t1.C_CUENTA = t2.C_CUENTA
      AND t1.AÑO = t2.AÑO
"""
res = work_conn.execute(text(sql_prelim_ratio))
_log("#ejec_a_prelim_dip_aper_r (prelim con Ratio)", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_ajtesp_ap"))
sql_ajtesp = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD,
       t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       'D63' AS C_SCN, 'S14' AS C_CAGENTE, '433' AS C_TRANSACCION,
       'transfesp' AS T_TRANSACCION, 'IG' AS N_CUENTA,
       t1.C_ACTIVO, '272' AS C_CUP, t1.C_ACT, t1.NIVEL,
       t1.DEVENGO * t1.Ratio AS DEVENGO
INTO #ejecucion_a_prelim_dip_ajtesp_ap
FROM #ejec_a_prelim_dip_aper_r t1
WHERE t1.Ratio IS NOT NULL
"""
res = work_conn.execute(text(sql_ajtesp))
_log("#ejecucion_a_prelim_dip_ajtesp_ap", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_aper"))
sql_prelim_neto = f"""
SELECT {cols_salida_t1},
       (CASE WHEN t1.Ratio IS NOT NULL THEN t1.DEVENGO * (1 - t1.Ratio)
             ELSE t1.DEVENGO END) AS DEVENGO
INTO #ejecucion_a_prelim_dip_aper
FROM #ejec_a_prelim_dip_aper_r t1
"""
work_conn.execute(text(sql_prelim_neto))
# PROC DATASETS APPEND BASE=... DATA=... FORCE -> INSERT server-side con columnas explícitas
# /*fin apertura de transf en especie de salud pública*/
sql_append_ajtesp = f"""
INSERT INTO #ejecucion_a_prelim_dip_aper ({cols_prelim})
SELECT {cols_prelim} FROM #ejecucion_a_prelim_dip_ajtesp_ap
"""
res = work_conn.execute(text(sql_append_ajtesp))
_log("APPEND transf. en especie -> #ejecucion_a_prelim_dip_aper", res.rowcount)


In [ ]:
res_ent = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_ENTIDAD IN (10201)"
))
res_32 = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim_dip_aper "
    "WHERE C_CUENTA = '32' AND C_ANALITICO IN ('04001000','04002000')"
))
# /*Analiticos predefinidos: AJUSTE COYUNTURA - se elimina de indicador transacción CORFO
#   que se desagrega anualmente entre ventas y D45 */
res_corfo = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim_dip_aper "
    "WHERE C_ENTIDAD IN (3801) AND C_CUENTA = '06' AND C_ITEM = '01' "
    "AND C_ASIGNACION IN ('00','000') AND C_ANALITICO IN ('01000000')"
))
_log("DELETE ajustes coyuntura #ejecucion_a_prelim_dip_aper",
     (res_ent.rowcount or 0) + (res_32.rowcount or 0) + (res_corfo.rowcount or 0))


In [ ]:
# /*Analiticos predefinidos (sigue). BASE DESGREGADA*/
# El GROUP BY del SAS no lleva ninguna función de resumen ni HAVING: PROC SQL lo
# degrada a ORDER BY (WARNING: 'A GROUP BY clause has been transformed into an
# ORDER BY clause') y devuelve TODAS las filas. Por eso acá no se agrupa ni se
# deduplica: es un SELECT 1:1 sobre el prelim, con DEVENGO renombrado a Dato.
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip_aper"))
sql_a_dip_aper = f"""
SELECT {cols_salida}, DEVENGO AS Dato
INTO #ejecucion_a_dip_aper
FROM #ejecucion_a_prelim_dip_aper
"""
res = work_conn.execute(text(sql_a_dip_aper))
_log("#ejecucion_a_dip_aper", res.rowcount)


In [ ]:
# /*Analiticos variables */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b_dip_aper"))
sql_b = f"""
SELECT {sel_ident_t1},
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       {case_c_act} AS C_ACT, t1.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000) AS Dato
INTO #ejecucion_b_dip_aper
FROM #ejec_dip_i t1
{join_scn2}
{join_prom_mes}
WHERE t1.C_CUENTA = '32'
  AND t2.C_SCN IN ({scn_coy})
  AND t1.C_ANALITICO IN ('04001000','04002000')
  AND t1.C_ENTIDAD NOT IN (5601,10201)
"""
work_conn.execute(text(sql_b))
res = work_conn.execute(text("DELETE FROM #ejecucion_b_dip_aper WHERE C_ENTIDAD IN (10201)"))
_log("DELETE #ejecucion_b_dip_aper C_ENTIDAD 10201", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c_dip_aper"))
# t2.OBS y t2.C_SCN vienen del LEFT JOIN: el blanco de SAS es NULL en SQL Server
# y NOT IN sobre NULL descartaría filas que el SAS conservaba
sql_c = f"""
SELECT {sel_ident_t1},
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       {case_c_act} AS C_ACT, t1.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000) AS Dato
INTO #ejecucion_c_dip_aper
FROM #ejec_dip_i t1
{join_scn2}
{join_prom_mes}
WHERE t1.C_CUENTA IN ('05','13','24','33')
  AND t2.C_SCN IN ({scn_coy})
  AND (t2.OBS IS NULL OR t2.OBS NOT IN ('CR18_difcoy','CR18_difact'))
  AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND (t2.C_SCN IS NULL OR t2.C_SCN NOT IN ('TC'))
"""
work_conn.execute(text(sql_c))
res = work_conn.execute(text("DELETE FROM #ejecucion_c_dip_aper WHERE C_ENTIDAD IN (10201)"))
_log("DELETE #ejecucion_c_dip_aper C_ENTIDAD 10201", res.rowcount)


In [ ]:
# /* Transacciones diferenciadas para coyuntura */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d_dip_aper"))
sql_d = f"""
SELECT {sel_ident_t1},
       t2.C_ACTIVO AS C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       {case_c_act} AS C_ACT, t1.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000) AS Dato
INTO #ejecucion_d_dip_aper
FROM #ejec_dip_i t1
{join_scn2}
{join_prom_mes}
WHERE t2.OBS IN ('CR18_difcoy')
  AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND t2.C_ACTIVO IN ({scn_coy})
  AND (t2.C_SCN IS NULL OR t2.C_SCN NOT IN ('TC'))
"""
res = work_conn.execute(text(sql_d))
_log("#ejecucion_d_dip_aper", res.rowcount)


In [ ]:
# /* Transacciones diferenciadas por actividad */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f_dip_aper"))
sql_f = f"""
SELECT {sel_ident_t1},
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       t2.C_CUP AS C_ACT, t1.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000) AS Dato
INTO #ejecucion_f_dip_aper
FROM #ejec_dip_i t1
{join_scn2}
{join_prom_mes}
WHERE t2.OBS IN ('CR18_difact')
  AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND (t2.C_SCN IS NULL OR t2.C_SCN NOT IN ('TC'))
  AND t2.C_SCN IN ({scn_coy})
"""
res = work_conn.execute(text(sql_f))
_log("#ejecucion_f_dip_aper", res.rowcount)


In [ ]:
# /*FORMATEA TABLA DE TCONS, PARA UNIRLA A LAS OTRAS*/
# C_ACTIVO='TC'; C_ACT=ACTIVIDAD; UPDATE C_ACT='AAPP' para la lista de actividades;
# y el subsetting IF C_SCN IN (...) del DATA step final
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_g_dip_aper"))
sql_g = f"""
SELECT SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA,
       C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO,
       C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION, N_CUENTA,
       'TC' AS C_ACTIVO, C_CUP,
       (CASE WHEN ACTIVIDAD IN ('FFAA','FFOO','PREVISION','FONASA','CORFO','CENABAST')
             THEN 'AAPP' ELSE ACTIVIDAD END) AS C_ACT,
       NIVEL, Dato
INTO #ejecucion_g_dip_aper
FROM #ejec_g_mixt_aper_dip_imp
WHERE C_SCN IN ({scn_coy})
"""
res = work_conn.execute(text(sql_g))
_log("#ejecucion_g_dip_aper", res.rowcount)


In [ ]:
# /*UNE TABLAS de Salida total de ejecución CG coy de data DIPRES*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_aper_prev"))
sql_prev = f"""
SELECT {cols_union} INTO #ejecucion_cg_dip_aper_prev FROM #ejecucion_a_dip_aper
UNION ALL SELECT {cols_union} FROM #ejecucion_b_dip_aper
UNION ALL SELECT {cols_union} FROM #ejecucion_c_dip_aper
UNION ALL SELECT {cols_union} FROM #ejecucion_d_dip_aper
UNION ALL SELECT {cols_union} FROM #ejecucion_g_dip_aper
UNION ALL SELECT {cols_union} FROM #ejecucion_f_dip_aper
"""
res = work_conn.execute(text(sql_prev))
_log("#ejecucion_cg_dip_aper_prev", res.rowcount)


In [ ]:
for _t in [
    "#ejecucion_a_prelim_dip_aper", "#ejecucion_a_dip_aper", "#ejecucion_b_dip_aper",
    "#ejecucion_c_dip_aper", "#ejecucion_d_dip_aper", "#ejecucion_g_dip_aper",
    "#ejecucion_f_dip_aper", "#ejec_a_prelim_dip_aper_0", "#ejec_a_prelim_dip_aper_r",
]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))
res = work_conn.execute(text("UPDATE #ejecucion_cg_dip_aper_prev SET N_CUENTA = 'DIPRES'"))
_log("UPDATE #ejecucion_cg_dip_aper_prev N_CUENTA='DIPRES'", res.rowcount)


In [ ]:
# /*Incorporacion de imputaciones permanentes*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cg_dip_aper_imput_perm"))
sql_imput_perm = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD,
       t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP,
       (CASE WHEN t1.C_ACT='FFAA' THEN 'AAPP'
             WHEN t1.C_ACT='FFOO' THEN 'AAPP'
             WHEN t1.C_ACT='PREVISION' THEN 'AAPP'
             WHEN t1.C_ACT='FONASA' THEN 'AAPP'
             WHEN t1.C_ACT='CORFO' THEN 'AAPP'
             WHEN t1.C_ACT='CENABAST' THEN 'AAPP'
             WHEN t1.C_ACT='AAPP' THEN 'AAPP'
             WHEN t1.C_ACT='SSPP' THEN 'SSPP'
             WHEN t1.C_ACT='EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL, t1.Dato
INTO #ejec_cg_dip_aper_imput_perm
FROM GOBGENER.dbo.IMPUT_GC_DIP_APER t1
WHERE t1.AÑO = {int(Year)}
  AND t1.C_SCN IN ({scn_coy})
"""
res = work_conn.execute(text(sql_imput_perm))
_log("#ejec_cg_dip_aper_imput_perm", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_aper_cob"))
sql_cob = f"""
SELECT {cols_union} INTO #ejecucion_cg_dip_aper_cob FROM #ejecucion_cg_dip_aper_prev
UNION ALL
SELECT {cols_union} FROM #ejec_cg_dip_aper_imput_perm
"""
res = work_conn.execute(text(sql_cob))
_log("#ejecucion_cg_dip_aper_cob", res.rowcount)


In [ ]:
# /*NUEVO: IMPUTA INGRESOS OPERACIONALES PARA LAS ENTIDADES QUE LO INFORMAN AGREGADO EN DIPRES, ASÍ SE MEJORA EL INDICADOR DE VENTAS*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip_07_cob"))
sql_a_07 = f"""
SELECT {sel_ident_t1},
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       {case_c_act} AS C_ACT, t1.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000) AS DEVENGO
INTO #ejecucion_a_dip_07_cob
FROM #ejec_dip_i t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
       ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
      AND t1.C_ANALITICO = t2.C_ANALITICO
{join_prom_mes}
WHERE t1.C_CUENTA IN ('07')
  AND t2.C_SCN IN ({scn_coy})
"""
res = work_conn.execute(text(sql_a_07))
_log("#ejecucion_a_dip_07_cob", res.rowcount)


In [ ]:
# /*Verifica que entidades informan solo el agregado para filtrarlas*/
work_conn.execute(text("DROP TABLE IF EXISTS #cuenta_item_07"))
sql_cuenta_item_07 = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD,
       t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       t1.C_CUP, COUNT(t1.DEVENGO) AS id
INTO #cuenta_item_07
FROM #ejecucion_a_dip_07_cob t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD,
         t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
         t1.C_CUP
"""
res = work_conn.execute(text(sql_cuenta_item_07))
_log("#cuenta_item_07", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_dip_07"))
sql_dip_07 = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD,
       t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION,
       'DIPRES' AS N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
       t1.DEVENGO AS Dato
INTO #ejecucion_dip_07
FROM #ejecucion_a_dip_07_cob t1
LEFT JOIN #cuenta_item_07 t2
       ON t1.MONEDA = t2.MONEDA AND t1.C_ENTIDAD = t2.C_ENTIDAD
      AND t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO
      AND t1.C_PROGRAMA = t2.C_PROGRAMA AND t1.C_TIPO_CUENTA = t2.C_TIPO_CUENTA
      AND t1.C_CUENTA = t2.C_CUENTA AND t1.MES = t2.MES
WHERE t2.id = 1
"""
res = work_conn.execute(text(sql_dip_07))
_log("#ejecucion_dip_07", res.rowcount)


In [ ]:
# /*une imputacion de ingresos operacionales*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_aper_imp"))
sql_imp = f"""
SELECT {cols_union}, N'Imputación cobertura' AS OBS
INTO #ejecucion_cg_dip_aper_imp
FROM #ejecucion_cg_dip_aper_cob
UNION ALL
SELECT {cols_union}, N'Imputación cobertura' AS OBS
FROM #ejecucion_dip_07
"""
res = work_conn.execute(text(sql_imp))
_log("#ejecucion_cg_dip_aper_imp", res.rowcount)


In [ ]:
res_cero = work_conn.execute(text("DELETE FROM #ejecucion_cg_dip_aper_imp WHERE DATO = 0"))
res_excl = work_conn.execute(text("DELETE FROM #ejecucion_cg_dip_aper_imp WHERE NIVEL = 'EXCLUIDO'"))
_log("DELETE #ejecucion_cg_dip_aper_imp (DATO=0 / NIVEL EXCLUIDO)",
     (res_cero.rowcount or 0) + (res_excl.rowcount or 0))


In [ ]:
# /*GENERA SALIDA AGRUPADA IMPUTACIONES POR COBERTURA DESDE DIPRES*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_imp"))
sql_cg_dip_imp = """
SELECT SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN,
       C_CAGENTE, C_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL,
       SUM(DATO) AS DATO, OBS
INTO #ejecucion_cg_dip_imp
FROM #ejecucion_cg_dip_aper_imp
GROUP BY SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN,
         C_CAGENTE, C_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, OBS
"""
work_conn.execute(text(sql_cg_dip_imp))
res = work_conn.execute(text("DELETE FROM #ejecucion_cg_dip_imp WHERE DATO = 0"))
_log("DELETE #ejecucion_cg_dip_imp DATO=0", res.rowcount)


In [ ]:
# DROP TABLE final del SAS (limpieza de WORK): solo sobre las temporales que
# existen en esta sesión
for _t in [
    "#ejec_cg_dip_aper_imput_perm", "#cuenta_item_07",
    "#ejecucion_a_prelim_dip_ajtesp_ap", "#ejecucion_dip_07",
    "#ejecucion_cg_dip_aper_prev", "#ratios_2",
]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## 4_Salida_Imputaciones

Consolida las imputaciones de cobertura de CGR y DIPRES —desagregadas y agregadas— y deja el detalle de entidades, periodos y fuente de cada imputación para auditoría

*confianza: medium · verificador: approve · SAS: DATA step SET (apilado) + PROC SQL CREATE TABLE con CASE y GROUP BY*

In [ ]:
# ========= 4_Salida_Imputaciones =========
# El DATA ... SET de SAS apila por NOMBRE de columna (y deja como faltante la
# que no existe en una de las tablas); el UNION ALL de T-SQL alinea por
# POSICIÓN, así que las columnas se resuelven en runtime y se alinean por nombre.
def _cols_temp(tabla):
    sql_cols = f"""
    SELECT name
    FROM tempdb.sys.columns
    WHERE object_id = OBJECT_ID('tempdb..{tabla}')
    ORDER BY column_id
    """
    return [fila[0] for fila in work_conn.execute(text(sql_cols)).fetchall()]


def _ramas_apiladas(tablas):
    cols = []
    cols_tabla = {}
    for t in tablas:
        cols_tabla[t] = _cols_temp(t)
        for c in cols_tabla[t]:
            if c not in cols:
                cols.append(c)
    ramas = []
    for t in tablas:
        proyeccion = ", ".join(
            f"[{c}]" if c in cols_tabla[t] else f"NULL AS [{c}]" for c in cols
        )
        ramas.append(f"SELECT {proyeccion} FROM {t}")
    return ramas


In [ ]:
# /*SALIDA IMPUTACIONES POR COBERTURA DESAGREGADAS CGR-DIPRES*/
_ramas_desag = _ramas_apiladas(["#ejecucion_cg_cgr_aper_imp", "#ejecucion_cg_dip_aper_imp"])
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cobertura_desag"))
sql_cobertura_desag = f"""
SELECT *
INTO #ejecucion_cobertura_desag
FROM (
{" UNION ALL ".join(_ramas_desag)}
) u
"""
res = work_conn.execute(text(sql_cobertura_desag))
_log("#ejecucion_cobertura_desag", res.rowcount)


In [ ]:
# /*detalla entidades imputadas y periodo*/
# CALCULATED FUENTE del PROC SQL: T-SQL no acepta el alias en el GROUP BY, se
# repite la expresión
work_conn.execute(text("DROP TABLE IF EXISTS #ent_cob_imp"))
sql_ent_cob_imp = """
SELECT C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, MONEDA, AÑO, MES,
       CASE WHEN N_CUENTA = 'DIPRES' THEN 'DIPRES' ELSE 'CGR' END AS FUENTE,
       COUNT(DATO) AS N
INTO #ent_cob_imp
FROM #ejecucion_cobertura_desag
GROUP BY MONEDA, AÑO, MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA,
         CASE WHEN N_CUENTA = 'DIPRES' THEN 'DIPRES' ELSE 'CGR' END
"""
work_conn.execute(text(sql_ent_cob_imp))
ent_cob_imp = pd.read_sql(
    text("SELECT * FROM #ent_cob_imp ORDER BY C_ENTIDAD, MONEDA, AÑO, MES"),
    work_conn,
)
_log("ent_cob_imp", ent_cob_imp)


In [ ]:
# /*SALIDA IMPUTACIONES POR COBERTURA AGREGADAS CGR-DIPRES*/
_ramas_ag = _ramas_apiladas(["#ejecucion_cg_cgr_imp", "#ejecucion_cg_dip_imp"])
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cobertura_ag"))
sql_cobertura_ag = f"""
SELECT *
INTO #ejecucion_cobertura_ag
FROM (
{" UNION ALL ".join(_ramas_ag)}
) u
"""
res = work_conn.execute(text(sql_cobertura_ag))
_log("#ejecucion_cobertura_ag", res.rowcount)
